In [ ]:
import tensorflow as tf
import sklearn
from keras.applications import EfficientNetV2B0, ResNet50, VGG16
from keras.optimizers import Adam
from keras.preprocessing import image_dataset_from_directory
from keras.losses import BinaryCrossentropy
import kagglehub
import os
from datasets import load_dataset

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ds = load_dataset("ILSVRC/imagenet-1k")
pretrained_imagenet = load_dataset("timm/mini-imagenet")

In [3]:
LEARNING_RATE=0.0001
BATCH_SIZE=64
IMAGE_SIZE=160
VAL_SPLIT=0.2

In [4]:
efficientnetv2b0 = EfficientNetV2B0(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
resnet50 = ResNet50(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
vgg16 = VGG16(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)

In [5]:
dataset_path = kagglehub.dataset_download("doctorstrange420/real-and-fake-ai-generated-art-images-dataset")
dataset_path

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1'

In [6]:
os.listdir(dataset_path+"\Data")

['FAKE', 'REAL']

In [7]:
fake_dir = os.path.join(dataset_path+"\Data", "FAKE")
fake_dir

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1\\Data\\FAKE'

In [8]:
real_dir = os.path.join(dataset_path+"/Data", "REAL")
real_dir

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1/Data\\REAL'

In [9]:
os.listdir(real_dir)[:5]

['00060d29813e54eec710cd6f9948a40ac.jpg',
 '000698a1cfe903e219f81178d8e7795cc.jpg',
 '00071f2b6fc24b1885ddb1b53b512081c.jpg',
 '000ad71148c3ad9bb8f68d78a0aeac70a.jpg',
 '000ad71148c3ad9bb8f68d78a0aeac70b.jpg']

In [10]:
os.listdir(fake_dir)[:5]

['img000006.jpg',
 'img000013.jpg',
 'img000014.jpg',
 'img000015.jpg',
 'img000016.jpg']

In [11]:
trainingDs = image_dataset_from_directory(dataset_path+"\Data", seed=123, subset="training", validation_split=VAL_SPLIT, batch_size=BATCH_SIZE, image_size=(IMAGE_SIZE, IMAGE_SIZE))
trainingDs

Found 21642 files belonging to 2 classes.
Using 17314 files for training.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [12]:
valDs = image_dataset_from_directory(dataset_path+"\Data", seed=123, subset="validation", validation_split=VAL_SPLIT, batch_size=BATCH_SIZE, image_size=(IMAGE_SIZE, IMAGE_SIZE))
valDs

Found 21642 files belonging to 2 classes.
Using 4328 files for validation.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [13]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1)
    ]
)
data_augmentation

<Sequential name=sequential, built=False>

In [14]:
vgg16.trainable=False

In [15]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.vgg16.preprocess_input(x)
x = vgg16.output
# x = vgg16(x, training=False)
x = tf.keras.layers.Flatten()(x)#GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [16]:
vgg16 = tf.keras.Model(inputs=vgg16.input, outputs=output)

In [17]:
vgg16.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 160, 160, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 160, 160, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 80, 80, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 80, 80, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 80, 80, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 40, 40, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 40, 40, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 40, 40, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 40, 40, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 20, 20, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 20, 20, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 20, 20, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 20, 20, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 10, 10, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 10, 10, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 10, 10, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 10, 10, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 5, 5, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,638,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,353,345 (62.38 MB)

 Trainable params: 1,638,657 (6.25 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [18]:
vgg16.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [19]:
efficientnetv2b0.trainable=False

In [20]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.efficientnet_v2.preprocess_input(x)
x = efficientnetv2b0.output
# x = efficientnetv2b0(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [21]:
efficientnetv2b0 = tf.keras.Model(inputs=efficientnetv2b0.input, outputs=output)

In [22]:
efficientnetv2b0.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 160, 160,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 160, 160,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 160, 160,  │          0 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 80, 80,    │        864 │ normalization[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 80, 80,    │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 80, 80,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 80, 80,    │      4,608 │ stem_activation[… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 80, 80,    │         64 │ block1a_project_… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 80, 80,    │          0 │ block1a_project_… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 40, 40,    │      9,216 │ block1a_project_… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 40, 40,    │        256 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 40, 40,    │          0 │ block2a_expand_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_co… │ (None, 40, 40,    │      2,048 │ block2a_expand_a… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_bn  │ (None, 40, 40,    │        128 │ block2a_project_… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_conv │ (None, 40, 40,    │     36,864 │ block2a_project_… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_bn   │ (None, 40, 40,    │        512 │ block2b_expand_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_act… │ (None, 40, 40,    │          0 │ block2b_expand_b

 Total params: 6,083,409 (23.21 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 5,919,312 (22.58 MB)

In [23]:
efficientnetv2b0.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [24]:
resnet50.trainable=False

In [25]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.resnet50.preprocess_input(x)
x = resnet50.output
# x = resnet50(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [26]:
resnet50 = tf.keras.Model(inputs=resnet50.input, outputs=output)

In [27]:
resnet50.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 160, 160,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 166, 166,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 80, 80,    │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 80, 80,    │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 80, 80,    │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 82, 82,    │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 40, 40,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 40, 40,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 40, 40,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 40, 40,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 40, 40,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 40, 40,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 40, 40,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 40, 40,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 40, 40,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 40, 40,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 40, 40,    │      1,024 │ conv2_block1_3_c

 Total params: 23,850,113 (90.98 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [28]:
resnet50.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [32]:
history = efficientnetv2b0.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=3
)

Epoch 1/3


271/271 ━━━━━━━━━━━━━━━━━━━━ 159s 588ms/step - accuracy: 0.7950 - loss: 0.4439 - precision_1: 0.7983 - recall_1: 0.7866 - val_accuracy: 0.8787 - val_loss: 0.3153 - val_precision_1: 0.8975 - val_recall_1: 0.8605
Epoch 2/3
271/271 ━━━━━━━━━━━━━━━━━━━━ 156s 577ms/step - accuracy: 0.8822 - loss: 0.2948 - precision_1: 0.8826 - recall_1: 0.8804 - val_accuracy: 0.9099 - val_loss: 0.2444 - val_precision_1: 0.9166 - val_recall_1: 0.9058
Epoch 3/3
271/271 ━━━━━━━━━━━━━━━━━━━━ 158s 582ms/step - accuracy: 0.9026 - loss: 0.2468 - precision_1: 0.9018 - recall_1: 0.9024 - val_accuracy: 0.9210 - val_loss: 0.2118 - val_precision_1: 0.9210 - val_recall_1: 0.9244


In [31]:
history = vgg16.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=3
)

Epoch 1/3
271/271 ━━━━━━━━━━━━━━━━━━━━ 821s 3s/step - accuracy: 0.7802 - loss: 0.8833 - precision: 0.7791 - recall: 0.7791 - val_accuracy: 0.8327 - val_loss: 0.4017 - val_precision: 0.8011 - val_recall: 0.8940
Epoch 2/3
271/271 ━━━━━━━━━━━━━━━━━━━━ 816s 3s/step - accuracy: 0.8901 - loss: 0.2667 - precision: 0.8864 - recall: 0.8938 - val_accuracy: 0.8618 - val_loss: 0.3612 - val_precision: 0.8639 - val_recall: 0.8655
Epoch 3/3
271/271 ━━━━━━━━━━━━━━━━━━━━ 801s 3s/step - accuracy: 0.9364 - loss: 0.1624 - precision: 0.9328 - recall: 0.9397 - val_accuracy: 0.8736 - val_loss: 0.3637 - val_precision: 0.8794 - val_recall: 0.8718


In [30]:
history = resnet50.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=3
    
)

Epoch 1/3
271/271 ━━━━━━━━━━━━━━━━━━━━ 512s 2s/step - accuracy: 0.8176 - loss: 0.3985 - precision_2: 0.8159 - recall_2: 0.8178 - val_accuracy: 0.9006 - val_loss: 0.2591 - val_precision_2: 0.9127 - val_recall_2: 0.8904
Epoch 2/3
271/271 ━━━━━━━━━━━━━━━━━━━━ 488s 2s/step - accuracy: 0.9016 - loss: 0.2423 - precision_2: 0.9006 - recall_2: 0.9018 - val_accuracy: 0.9242 - val_loss: 0.2051 - val_precision_2: 0.9320 - val_recall_2: 0.9185
Epoch 3/3
271/271 ━━━━━━━━━━━━━━━━━━━━ 466s 2s/step - accuracy: 0.9239 - loss: 0.1920 - precision_2: 0.9213 - recall_2: 0.9262 - val_accuracy: 0.9314 - val_loss: 0.1782 - val_precision_2: 0.9260 - val_recall_2: 0.9407
